In [ ]:
### Load full-study datasets and perform Mantel tests
library(linkET)
suppressPackageStartupMessages(library(tidyverse))

yield <- read_csv("../results/book/2/yield_dataset.csv", show_col_types = FALSE)
feature <- read_csv("../results/book/2/feat_dataset.csv", show_col_types = FALSE)

### Calculate Mantel statistics and significance groups
mantel <- mantel_test(yield, feature,
    spec_dist = "euclidean",
    env_dist = "euclidean",
    spec_select = list("YO" = 1, "TY" = 2, "DY" = 3),
    permutations = 999
) %>%
    mutate(
        rd = cut(abs(r),
            breaks = c(-Inf, 0.15, Inf),
            labels = c("<= 0.15", "> 0.15")
        ),
        pd = cut(p,
            breaks = c(-Inf, 0.005, 0.01, 0.05, Inf),
            labels = c("<= 0.005", "0.005 - 0.01", "0.01 - 0.05", "> 0.05")
        )
    )
write_csv(mantel, "../results/book/2/mantel_result.csv")


In [ ]:
### Compute feature correlations and generate the full-study network plot
library(linkET)
library(ggnewscale)
library(RColorBrewer)
suppressPackageStartupMessages(library(viridis))
suppressPackageStartupMessages(library(tidyverse))
options(lifecycle_verbosity = "quiet")

feature <- read_csv("../results/book/2/feat_dataset.csv", show_col_types = FALSE)
mantel <- read_csv("../results/book/2/mantel_result.csv", show_col_types = FALSE) %>%
    mutate(pd = factor(pd, levels = c("<= 0.005", "0.005 - 0.01", "0.01 - 0.05", "> 0.05")))

### Compute and export the Pearson correlation matrix
cor_mat <- correlate(feature)
cor_mat <- unclass(cor_mat)
cor_mat <- as.data.frame(cor_mat)
cor_mat <- tibble::rownames_to_column(cor_mat, var = "Var")
write_csv(cor_mat, "../results/book/2/correl_matrix.csv")

### Combine Pearson correlations and Mantel associations in one plot
p2 <- qcorrplot(correlate(feature), type = "upper", diag = FALSE, grid_col = NA) +
    geom_point(shape = 21, size = 4, fill = NA, stroke = 0.5, color = "black") +
    geom_point(aes(size = abs(r), fill = r),
        shape = 21,
        stroke = 0.4,
        color = "black"
    ) +
    scale_size(range = c(1, 3), guide = "none") +
    new_scale("size") +
    geom_couple(
        data = mantel,
        aes(color = pd, size = rd),
        label.size = 2.46,
        label.family = "Arial",
        label.fontface = 2,
        nudge_x = 0.5,
        curvature = nice_curvature(by = "from")
    ) +
    scale_fill_gradientn(
        limits = c(-0.8, 0.8),
        breaks = seq(-0.8, 0.8, 0.4),
        colors = rev(brewer.pal(11, "RdBu")),
        oob = scales::squish
    ) +
    scale_size_manual(values = c(0.3, 1)) +
    scale_color_manual(values = viridis(8, alpha = 0.88)) +
    guides(
        fill = guide_colorbar(
            title = "Pearson's r",
            raster = FALSE,
            title.vjust = 3,
            keyheight = unit(1.8, "cm"),
            keywidth = unit(0.3, "cm"),
            order = 1
        ),
        size = guide_legend(
            title = "Mantel's r",
            order = 2,
            keyheight = unit(0.3, "cm")
        ),
        colour = guide_legend(
            title = "Mantel's p",
            order = 3,
            keyheight = unit(0.3, "cm")
        )
    ) +
    theme(
        legend.box.spacing = unit(3, "pt"),
        axis.text = element_text(size = 6),
        legend.title = element_text(size = 6),
        legend.text = element_text(size = 5)
    )
### Export the network plot as an SVG file
ggsave(p2, file = "../results/book/2/net_heat_plot.svg", width = 5, height = 4)
